## Dynamic Operability of a CSTR Under Model Uncertainty (GP-NARX Surrogate)

Author: Victor Alves - Carnegie Mellon University

When the process model is learned from data instead of derived from first principles, the dynamic operability funnel inherits the model's uncertainty. This case study trains a **Gaussian process (GP)** surrogate on input-output data from a CSTR and propagates the surrogate's predictive uncertainty through the dynamic operability analysis.

A GP returns, at every query point, a mean prediction and a standard deviation $\sigma$ that grows where training data is sparse {cite}`rasmussen2008`; using such a surrogate inside an operability analysis follows the Gaussian-process operability framework of {cite}`alves2022gp`. The surrogate here is a **GP-NARX** model: it predicts the next state from the current state and input, $x_{k+1} = \mu_{GP}(x_k, u_k) \pm \sigma_{GP}(x_k, u_k)$, which is exactly the step model the dynamic operability framework needs {cite}`dinh23`. Mapping the funnel three times, once at the mean and once at each of the $\pm 3\sigma$ bounds, turns the surrogate's uncertainty into a **band of achievable output funnels**: the reactor's reachable region is no longer a single funnel but a range consistent with what the data support.

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

# Plotly renderer that stays interactive in the compiled
# documentation website (loads plotly.js via require.js).
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

### The CSTR

The data-generating plant is the nonisothermal CSTR with a first-order irreversible reaction $A \to B$, the same reactor used in the [collocation tutorial](dynamic_pyomo_collocation.ipynb). Its states are the dimensionless concentration $x_1$ and temperature $x_2$, and its inputs are the feed concentration $u_1$ and coolant temperature $u_2$:

$$\frac{dx_1}{dt} = \frac{1}{\theta}(u_1 - x_1) - Da\, x_1 \exp\!\left(\frac{x_2}{1 + x_2/\gamma}\right)$$

$$\frac{dx_2}{dt} = -\frac{x_2}{\theta} + B\, Da\, x_1 \exp\!\left(\frac{x_2}{1 + x_2/\gamma}\right) - \beta(x_2 - u_2)$$

with $Da = 0.072$, $\gamma = 20$, $B = 8$, $\beta = 0.3$, and $\theta = 20$. The outputs are the two states. Here this model stands in for an experiment or a high-fidelity simulator: it is sampled to produce training data, and the GP surrogate learned from that data is what the operability analysis actually uses.

In [2]:
# CSTR parameters
Da, gamma, B_rxn, beta, theta = 0.072, 20.0, 8.0, 0.3, 20.0
dt = 1.0  # step length [dimensionless time]


def cstr_rhs(t, x, u):
    """
    Right-hand side of the nonisothermal CSTR, dx/dt = f(x, u).

    Parameters
    ----------
    t : float
        Time (unused; the dynamics are autonomous).
    x : array-like, shape (2,)
        State ``[concentration, temperature]``.
    u : array-like, shape (2,)
        Inputs ``[feed concentration, coolant temperature]``.

    Returns
    -------
    list of float
        The derivative ``[dx1/dt, dx2/dt]``.
    """
    x1, x2 = x
    rate = Da * x1 * np.exp(x2 / (1.0 + x2 / gamma))
    dx1 = (1.0 / theta) * (u[0] - x1) - rate
    dx2 = -(x2 / theta) + B_rxn * rate - beta * (x2 - u[1])
    return [dx1, dx2]


def cstr_step(x, u):
    """
    Advance the true CSTR by one step of length ``dt`` with scipy integration.

    This is the data-generating plant: it is sampled to train the GP surrogate.

    Parameters
    ----------
    x : array-like, shape (2,)
        Current state.
    u : array-like, shape (2,)
        Inputs held constant over the step.

    Returns
    -------
    x_next : numpy.ndarray, shape (2,)
        State at the end of the step.
    y : numpy.ndarray, shape (2,)
        Output, here the identity ``y = x_next``.
    """
    sol = solve_ivp(cstr_rhs, [0.0, dt], np.asarray(x, dtype=float),
                    args=(np.asarray(u, dtype=float),),
                    method='RK45', rtol=1e-8)
    x_next = sol.y[:, -1]
    return x_next, x_next


# Relax the reactor to its steady state at the nominal inputs; the funnel is
# mapped outward from this operating point.
u_nom = np.array([1.0, 0.0])
x0 = solve_ivp(cstr_rhs, [0.0, 2000.0], [1.0, 0.0],
               args=(u_nom,), method='LSODA', rtol=1e-10).y[:, -1]

AIS_bounds = np.array([[0.8, 1.2],     # u1: feed concentration
                       [-0.5, 0.5]])   # u2: coolant temperature
DOS_bounds = np.array([[0.1, 0.4],     # y1: concentration
                       [0.0, 2.0]])    # y2: temperature
k_max = 12

print(f'Nominal steady state: x1 = {x0[0]:.3f}, x2 = {x0[1]:.3f}')

Nominal steady state: x1 = 0.230, x2 = 0.880


### The GP-NARX surrogate

We sample states and inputs across the operating region, advance the true CSTR one step, and add a little measurement noise to mimic real measurements. One GP is trained per output to predict the next state from the current state and input (a first-order nonlinear autoregressive model with exogenous input, GP-NARX). Each GP returns a posterior mean and a standard deviation $\sigma$; $\sigma$ is small where data is dense and grows where it is sparse, which is how the surrogate reports its own uncertainty.

In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

# Sample (state, input) points over the operating region and record the true
# next state, with a small amount of measurement noise.
rng = np.random.RandomState(1)
n_train = 50
X_train = rng.uniform([0.10, -0.10], [0.40, 2.10], (n_train, 2))
U_train = rng.uniform(AIS_bounds[:, 0], AIS_bounds[:, 1], (n_train, 2))
X_next = np.array([cstr_step(X_train[i], U_train[i])[0]
                   for i in range(n_train)])
X_next += rng.normal(0.0, 2e-3, X_next.shape)
features = np.hstack([X_train, U_train])

# Standardize the four features so the GP length scales are well conditioned.
scaler = StandardScaler().fit(features)
features_scaled = scaler.transform(features)

# One GP per output: an automatic-relevance-determination squared-exponential
# kernel plus a white-noise term for the measurement noise.
gp_models = []
for d in range(2):
    kernel = (ConstantKernel(1.0) * RBF(length_scale=[1.0] * 4)
              + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1)))
    gp = GaussianProcessRegressor(kernel=kernel,
                                  n_restarts_optimizer=4,
                                  normalize_y=True,
                                  random_state=1)
    gp.fit(features_scaled, X_next[:, d])
    gp_models.append(gp)
    rmse = np.sqrt(np.mean((gp.predict(features_scaled) - X_next[:, d]) ** 2))
    print(f'GP for x{d + 1}: training RMSE = {rmse:.4f}')


def gp_step_model(gp_models, scaler, n_sigma):
    """
    Build a step model from the GP posterior, offset by ``n_sigma`` standard
    deviations.

    With ``n_sigma = 0`` the step model follows the GP posterior mean; positive
    and negative values shift each predicted next state by that many GP standard
    deviations, giving the optimistic and pessimistic bounds that bracket the
    funnel.

    Parameters
    ----------
    gp_models : list of GaussianProcessRegressor
        One fitted GP per output, each predicting the next state from the
        scaled (state, input) features.
    scaler : sklearn.preprocessing.StandardScaler
        The fitted scaler applied to the (state, input) features before
        prediction.
    n_sigma : float
        Number of posterior standard deviations to offset the prediction by.

    Returns
    -------
    Callable
        A step model ``step(x, u) -> (x_next, y)`` with ``y = x_next``,
        matching the contract expected by ``dynamic_operability``.
    """
    def step(x, u):
        feat = scaler.transform(np.hstack([np.asarray(x),
                                           np.asarray(u)]).reshape(1, -1))
        x_next = np.empty(len(gp_models))
        for d, gp in enumerate(gp_models):
            mu, sigma = gp.predict(feat, return_std=True)
            x_next[d] = mu[0] + n_sigma * sigma[0]
        return x_next, x_next
    return step

GP for x1: training RMSE = 0.0016
GP for x2: training RMSE = 0.0012


### Mapping the funnel under uncertainty

Mapping the funnel with the GP mean gives the nominal achievable output set; mapping it with the $+3\sigma$ and $-3\sigma$ step models gives the optimistic and pessimistic bounds the data still support. Overlaid, the three funnels form a band, and the wider the band, the more the limited data leave the reactor's reachable region uncertain. The state is low-dimensional, so ``dynamic_operability`` maps each funnel by nonlinear state-space projection {cite}`dinh23`.

Importing opyrability's dynamic operability and funnel comparison functions:

In [4]:
from opyrability import dynamic_operability, plot_funnel_comparison

# Map the funnel at the GP mean and at the +/-3 sigma bounds.
funnels = {}
for label, n_sigma in [('GP mean', 0.0),
                       ('GP +3 sigma', 3.0),
                       ('GP -3 sigma', -3.0)]:
    model = gp_step_model(gp_models, scaler, n_sigma)
    funnels[label] = dynamic_operability(model,
                                         x0,
                                         AIS_bounds,
                                         DOS=DOS_bounds,
                                         k_max=k_max,
                                         method='projection',
                                         AIS_resolution=3,
                                         plot=False)

doi = {label: res['dOI'][-1] for label, res in funnels.items()}
print(f"Final dOI: mean = {doi['GP mean']:.1f} %, "
      f"+3 sigma = {doi['GP +3 sigma']:.1f} %, "
      f"-3 sigma = {doi['GP -3 sigma']:.1f} %")
print(f"Uncertainty band width: "
      f"{doi['GP +3 sigma'] - doi['GP -3 sigma']:.1f} percentage points")

fig, _ = plot_funnel_comparison(funnels,
                                labels=['Concentration x<sub>1</sub>',
                                        'Temperature x<sub>2</sub>'],
                                colors=['black', 'firebrick', 'royalblue'],
                                orientation='vertical',
                                engine='plotly',
                                title='CSTR achievable output funnel under '
                                      'GP uncertainty (mean and ±3σ)')

Final dOI: mean = 45.3 %, +3 sigma = 51.7 %, -3 sigma = 38.5 %
Uncertainty band width: 13.2 percentage points


### Conclusions

**Key Results:**

- A GP-NARX surrogate trained on 50 noisy samples reproduces the CSTR's one-step dynamics to within a small training RMSE, and its posterior standard deviation provides a built-in uncertainty estimate.
- Mapping the dynamic funnel at the GP mean and at the $\pm 3\sigma$ bounds turns that uncertainty into a band of achievable output funnels: the final dOI spans roughly $39\%$ to $52\%$ of the DOS, an uncertainty of about $13$ percentage points attributable to the limited training data.
- Because the surrogate respects the step-model contract, the same ``dynamic_operability`` call used for first-principles models applies unchanged to a data-driven one.

**Workflow Summary:**

1. Sample the plant and train one GP per output (GP-NARX).
2. Build the mean and $\pm 3\sigma$ step models from the GP posterior.
3. Map the funnel for each with ``dynamic_operability`` and overlay them with ``plot_funnel_comparison``.